# Clustering-Based Text Summarization Using LangChain’s Map-Reduce Method

### Introduction
The surge in open-ended feedback data — from customer reviews to survey responses — demands tools that can interpret, synthesize, and summarize vast volumes of unstructured text. Traditional extractive or abstractive summarization techniques often generate generalized summaries that miss nuances buried in scattered opinions. A promising alternative is to combine semantic clustering with LangChain’s map-reduce summarization method, allowing us to derive granular, theme-driven summaries that retain contextual relevance.

In this article, we explore a hybrid approach that clusters semantically similar responses before applying LangChain’s map-reduce summarization. This enables us to generate summaries that are not only concise but also thematically organized and scalable for large datasets.

### Motivation
Consider a scenario where you collect 10,000 customer responses from a post-purchase survey. A single summary of all these responses could dilute the richness of individual sentiments. Instead, grouping similar feedback first (e.g., complaints about delivery vs. praise for customer support) and then summarizing each group can lead to more actionable insights.

This approach is particularly valuable for:
- Customer experience teams looking to categorize and act on feedback.
- Researchers analyzing qualitative responses.
- Product managers summarizing user pain points across categories.
### High-Level Workflow
1. **Text Embedding**: Convert free-text responses into vector embeddings using a sentence transformer.
2. **Clustering**: Group similar responses using KMeans, HDBSCAN or Agglomerative Clustering.
3. **Map Step**: Generate summaries for each cluster using LangChain.
4. **Reduce Step**: Combine those cluster-wise summaries into a global summary.
5. **Optional — Theme Naming**: Assign human-readable themes to each cluster using LLMs or keyword extraction.
Press enter or click to view image in full size

### Step-by-Step Implementation

1. Environment Setup

In [ ]:
pip install sentence-transformers langchain openai hdbscan

2. Load and Embed the Responses

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
responses = [
    "The checkout process was confusing and long.",
    "Customer support was very helpful.",
    "I loved the fast delivery.",
    "The website UI needs improvement.",
    "Delivery was quick and seamless.",
    "Support team resolved my issue in minutes."
]
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(responses)


3. Clustering the Responses

In [ ]:
import hdbscan
from collections import defaultdict

In [ ]:
clusterer = hdbscan.HDBSCAN(min_cluster_size=2, metric='euclidean')
clusters = clusterer.fit_predict(embeddings)
clustered_responses = defaultdict(list)
for idx, cluster_id in enumerate(clusters):
    if cluster_id != -1:
        clustered_responses[cluster_id].append(responses[idx])

Now, each cluster contains semantically similar responses.

4. LangChain Map-Reduce Summarization

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.chains.summarize import load_summarize_chain
from langchain.docstore.document import Document

In [ ]:
llm = ChatOpenAI(temperature=0, model_name="gpt-4")
cluster_summaries = []
for cluster_id, texts in clustered_responses.items():
    documents = [Document(page_content=text) for text in texts]
    chain = load_summarize_chain(llm, chain_type="map_reduce")
    summary = chain.run(documents)
    cluster_summaries.append(summary)

5. Reduce Step: Final Global Summary

In [ ]:
final_docs = [Document(page_content=s) for s in cluster_summaries]
final_summary_chain = load_summarize_chain(llm, chain_type="stuff")
final_summary = final_summary_chain.run(final_docs)

In [ ]:
print("Final Summary:\n", final_summary)

Sample Output

In [ ]:
Final Summary:
Customers appreciated fast delivery and helpful customer support. However, some noted issues with the website interface and the checkout experience.

### Theme Labeling (Optional)
You can label each cluster with a descriptive theme using an LLM:

In [ ]:
theme_prompt = """
Given the following list of user responses:
{responses}
Identify a short, clear theme or topic that represents all of them.
"""

In [ ]:
for cluster_id, texts in clustered_responses.items():
    joined_text = "\n".join(texts)
    theme = llm.predict(theme_prompt.format(responses=joined_text))
    print(f"Cluster {cluster_id} Theme: {theme}")

#### Benefits of This Approach
- Thematic Summarization: Captures context-specific insights.
- Scalability: Efficiently handles large volumes of text.
- Interpretability: Provides clarity on what each group of responses represents.
- Modular: Swap models or clustering methods without altering the core logic.
#### Challenges and Considerations
- Cluster Quality: Requires tuning (e.g., min_cluster_size) to avoid over/under clustering.
- LLM Costs: Summarizing clusters can be expensive with large volumes.
- Theme Naming Accuracy: May need human verification.
#### Conclusion
This clustering-based summarization pipeline leverages the power of semantic embeddings and LangChain’s map-reduce framework to produce high-quality, interpretable summaries. It’s particularly well-suited for analyzing open-ended text data in customer research, social listening, and employee surveys.

With a few modifications, this pipeline can also be extended to multilingual datasets, incorporate sentiment analysis, or support real-time feedback summarization.